# Beyond-Flesch ELECTRA Gradio Demo

Standalone Colab notebook for inference only. It loads the saved ELECTRA + ScalarMix DANN checkpoint from Google Drive and launches a Gradio UI.

Default checkpoint: `best_model.pt` from the CNN + OneStop + RACE training run, because that checkpoint had the best overall OOD score in your latest comparison.

Inputs: sentence, paragraph, or passage. Output: `elementary`, `middle`, or `high` with probabilities.

In [ ]:
!pip install -q transformers torch gradio

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# ── Config ─────────────────────────────────────────────────────────────────
DRIVE_OUT_DIR = "/content/drive/MyDrive/BeyondFK/trail/electra_cnn_ose_race_train_multi_ood_dann"

# Use best_model.pt for the best validation / aggregate OOD checkpoint.
# Change to "phase2_final.pt" if you specifically want DANN-final weights.
CKPT_NAME = "best_model.pt"

MODEL_NAME = "google/electra-large-discriminator"
MAX_LEN = 512

label2id = {"elementary": 0, "middle": 1, "high": 2}
id2label = {v: k for k, v in label2id.items()}

CKPT_PATH = f"{DRIVE_OUT_DIR}/{CKPT_NAME}"
print("Checkpoint:", CKPT_PATH)

In [ ]:
# ── Model definition: same architecture used during training ───────────────
from __future__ import annotations

from typing import Any, List, Tuple

import os
import torch
import torch.nn as nn
from torch import Tensor
from torch.nn import Parameter, ParameterList
from transformers import AutoModel, AutoTokenizer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


class ScalarMix(nn.Module):
    def __init__(self, mixture_size: int, trainable: bool = True) -> None:
        super().__init__()
        self.scalar_parameters = ParameterList(
            [Parameter(torch.zeros(1), requires_grad=trainable) for _ in range(mixture_size)]
        )
        self.gamma = Parameter(torch.ones(1), requires_grad=trainable)

    def forward(self, tensors: List[torch.Tensor]) -> torch.Tensor:
        w = torch.nn.functional.softmax(torch.cat([p for p in self.scalar_parameters]), dim=0)
        w = torch.split(w, 1)
        return self.gamma * sum(weight * t for weight, t in zip(w, tensors))


class GradientReversalFunction(torch.autograd.Function):
    @staticmethod
    def forward(ctx: Any, x: Tensor, lambda_: float) -> Tensor:
        ctx.lambda_ = float(lambda_)
        return x.view_as(x)

    @staticmethod
    def backward(ctx: Any, grad_output: Tensor) -> Tuple[Tensor, None]:
        return -ctx.lambda_ * grad_output, None


def apply_gradient_reversal(x: Tensor, lambda_: float) -> Tensor:
    return GradientReversalFunction.apply(x, float(lambda_))


class DifficultyClassifierHead(nn.Module):
    def __init__(self, in_dim: int, num_classes: int = 3, dropout: float = 0.1) -> None:
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 256), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(256, num_classes),
        )

    def forward(self, x: Tensor) -> Tensor:
        return self.net(x)


class DomainClassifierHead(nn.Module):
    def __init__(self, in_dim: int, num_domains: int, dropout: float = 0.1) -> None:
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 256), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(256, num_domains),
        )

    def forward(self, x: Tensor) -> Tensor:
        return self.net(x)


class ElectraScalarMixDANN(nn.Module):
    def __init__(self, model_name: str, num_classes: int, num_domains: int, dropout: float = 0.2) -> None:
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden = int(self.encoder.config.hidden_size)
        n_layers = int(self.encoder.config.num_hidden_layers) + 1
        self.scalar_mix = ScalarMix(n_layers)
        self.dropout = nn.Dropout(dropout)
        self.difficulty_head = DifficultyClassifierHead(hidden, num_classes, dropout)
        self.domain_head = DomainClassifierHead(hidden, num_domains, dropout)

    def encode_pooled(self, input_ids: Tensor, attention_mask: Tensor) -> Tensor:
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask, output_hidden_states=True)
        mixed = self.dropout(self.scalar_mix(list(out.hidden_states)))
        mask = attention_mask.unsqueeze(-1).float()
        return (mixed * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1e-9)

    def forward(self, input_ids: Tensor, attention_mask: Tensor, grl_lambda: float) -> Tuple[Tensor, Tensor]:
        pooled = self.encode_pooled(input_ids, attention_mask)
        diff_logits = self.difficulty_head(pooled)
        dom_logits = self.domain_head(apply_gradient_reversal(pooled, grl_lambda))
        return diff_logits, dom_logits

    def difficulty_logits_only(self, input_ids: Tensor, attention_mask: Tensor) -> Tensor:
        return self.difficulty_head(self.encode_pooled(input_ids, attention_mask))

In [ ]:
# ── Load checkpoint from Drive ─────────────────────────────────────────────
assert os.path.exists(CKPT_PATH), f"Missing checkpoint: {CKPT_PATH}"

ckpt = torch.load(CKPT_PATH, map_location=device)
domain2id = ckpt.get("domain2id", {"default": 0})
num_domains = len(domain2id)

model = ElectraScalarMixDANN(MODEL_NAME, num_classes=3, num_domains=num_domains).to(device)
model.load_state_dict(ckpt["state_dict"])
model.eval()

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Loaded:", CKPT_PATH)
print("Checkpoint phase:", ckpt.get("phase", "best_val"))
print("Domains:", domain2id)

In [ ]:
# ── Gradio app ─────────────────────────────────────────────────────────────
import gradio as gr
import torch.nn.functional as F


def classify_reading_level(text: str):
    text = str(text).strip()
    if not text:
        return {"elementary": 0.0, "middle": 0.0, "high": 0.0}

    enc = tokenizer(
        text,
        truncation=True,
        max_length=MAX_LEN,
        padding=True,
        return_tensors="pt",
    )
    enc = {k: v.to(device) for k, v in enc.items()}

    with torch.no_grad():
        logits = model.difficulty_logits_only(enc["input_ids"], enc["attention_mask"])
        probs = F.softmax(logits, dim=-1).squeeze(0).detach().cpu().numpy()

    return {id2label[i]: float(probs[i]) for i in range(len(probs))}


demo = gr.Interface(
    fn=classify_reading_level,
    inputs=gr.Textbox(
        lines=8,
        placeholder="Paste a sentence, paragraph, or passage here...",
        label="Input text",
    ),
    outputs=gr.Label(num_top_classes=3, label="Predicted education level"),
    title="Beyond-Flesch Reading Level Classifier",
    description=(
        "ELECTRA + ScalarMix classifier trained on Llama-judge labels. "
        "Predicts elementary, middle, or high school reading level."
    ),
    examples=[
        ["The cat sat on the mat. It was happy and warm."],
        ["Climate change affects ecosystems, agriculture, and public health across the world."],
        ["The legislature ratified the constitutional amendment after prolonged bipartisan negotiations."],
    ],
)

demo.launch(share=True)